
# Virtual friend

This notebook builds an advanced chatbot with:
✅ **Ollama LLM** integration  
✅ **Streaming responses**  
✅ **Conversation memory (buffer + summary + persistent storage)**  
✅ **Optional Web Search** (via SerpAPI or similar)  

---


## 📥 Imports and Configuration

In [ ]:

import os ,logging ,json
import pyttsx3
from sk_tools import AgentTools
from sk_listen import ListenAudio
from sk_speak import SkSpeaker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory, CombinedMemory
from langchain.chains import LLMChain
from langchain.agents import initialize_agent, Tool
from langchain.utilities import SerpAPIWrapper
from langchain.agents import AgentType
from murf import Murf
import pygame


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Ollama configuration
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://'your ipv4' :11434")
MAIN_MODEL = os.getenv("OLLAMA_MAIN_MODEL", "gemma2:2b")
SUMMARY_MODEL = os.getenv("OLLAMA_SUMMARY_MODEL", "phi3")
# client = Murf(api_key="YOUR_API_KEY")



In [ ]:
llm = OllamaLLM(model=MAIN_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.2, top_p=0.9, streaming=True)
summary_llm = OllamaLLM(model=SUMMARY_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.2, top_p=0.9, streaming=False)


## 🧠 Prompt Template & Memory

In [10]:

system_prompt = """
You are **Sk**, a friendly, curious, and knowledgeable femail AI companion and you are a teenager.  
You have access to several helpful tools that you can use whenever they are needed to answer a user’s question, or whenever the user explicitly asks you to use them.  

Your style:  
✅ Explain things in **clear, simple language** so anyone can understand.  
✅ Use **examples or analogies** to make concepts easy and fun.  
✅ Keep a **positive and enthusiastic tone**—like a friend excited to help.  
✅ If a question has multiple parts, handle them step by step, and combine results into a smooth, helpful answer.  
✅ If a tool is needed (like search, calculator, or others), don’t hesitate to use it. 
✅ Don't use emogies

Your goal is to educate, help, and spark curiosity while keeping the conversation lively and enjoyable!
"""

template = f"""
System Prompt:
{system_prompt}

Past conversation:
{{message_buffer_log}}

Conversation summary:
{{message_summary_log}}

Question: {{question}}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

# Memory
buffer_memory = ConversationBufferMemory(memory_key="message_buffer_log", input_key="question", return_messages=False)
summary_memory = ConversationSummaryMemory(llm=summary_llm, memory_key="message_summary_log", input_key="question", return_messages=False)
combined_memory = CombinedMemory(memories=[summary_memory, buffer_memory])


## Tools for agent

In [11]:
tools =AgentTools.get_all_tools(include_search=True)

## 🔗 Chat Chain and Functions

In [12]:
chat_chain = LLMChain(llm=llm, prompt=prompt, memory=combined_memory, output_parser=StrOutputParser())
# chat_chain = prompt | llm | combined_memory | StrOutputParser()

# Persistent memory file
MEMORY_FILE = "chat_memory.json"

def save_memory():
    with open(MEMORY_FILE, 'w') as f:
        json.dump({
            "buffer": buffer_memory.load_memory_variables({}),
            "summary": summary_memory.load_memory_variables({})
        }, f)

def load_memory():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, 'r') as f:
            data = json.load(f)
            # In practice, you'd restore states carefully; here we just log
            logger.info("Loaded previous memory (not restoring into objects).")

def ask_question(question: str, use_stream: bool = True, use_tools: bool = True) -> str:
    global tools
    # logger.info(f"Asking: {question}")
    tools = tools
    agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)
    if use_tools:
        return  agent.invoke({"input":question})
    if use_stream:
        response = ""
        for chunk in chat_chain.stream({"question": question}):
            text = chunk.get("text", "")
            print(text, end="", flush=True)
            response += text
        print()
        save_memory()
        return response
    else:
        result = chat_chain.invoke({"question": question})
        save_memory()
        return result["text"]

load_memory()


INFO:__main__:Loaded previous memory (not restoring into objects).


In [14]:
# print(agent.run("open youtube and play galtis se mistake "))

In [15]:
# print(agent.run("play mere saamne wali khidki me ek chaad ka tukda rhaha ta hai na on youtube and skip any add"))

# listen and response(audio) function

In [ ]:
# Initialize
speaker = SkSpeaker()
listener = ListenAudio(wake_word="jarvis")

# Optional: choose a voice
# speaker.list_voices()
speaker.set_voice(1)

# Callback when wake word is detected
def on_wake_command(query: str):
    command = query.replace("jarvis", "").strip()
    if not command:
        command = "Yes?"
    print(f"[Jarvis] Command received: {command}")

    # 🤖 Get response from your agent
    response = ask_question(command)  # your custom function
    speaker.speak(response)
# Start listening in background
listener.start_listening(on_wake_command)

[SkSpeaker] Ready with OS voices.


🎤 Listening continuously... (say wake word)
🗣️ Heard: hi jarvis
✅ Wake word 'jarvis' detected!
[Jarvis] Command received: hi


> Entering new AgentExecutor chain...
⚠️ Error: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond
